# Task 1

In [140]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r'c:\Users\HP\OneDrive\Desktop\download')
print(pd.read_sql_query("SELECT * FROM members", conn))
print('-----')
print(pd.read_sql_query("SELECT * FROM books", conn))
print('-----')
print(pd.read_sql_query("SELECT * FROM checkouts", conn))
print('======')

# query='''
# SELECT members.member_id, members.first_name, COUNT(*) AS borrows_count
# FROM checkouts
# LEFT JOIN members
# ON checkouts.member_id = members.member_id
# GROUP BY checkouts.member_id
# '''
# print(pd.read_sql_query(query, conn)) #How much is each member borrowing?

# checkouts = pd.read_sql_query('SELECT * FROM checkouts', conn)
# print(checkouts['member_id'].value_counts()) #How much is each member borrowing?

print(pd.read_sql_query("SELECT title, author FROM books WHERE author LIKE 'H%'", conn)) #Author Patter: name starting with "H"

query = '''
SELECT books.book_id, books.title, COUNT(*) AS borrows_count
FROM checkouts
JOIN books
ON checkouts.book_id = books.book_id
GROUP BY checkouts.book_id
ORDER BY COUNT(*) DESC
LIMIT 5
'''
print(pd.read_sql_query(query, conn)) #What are the most popular books?

query = '''
SELECT members.member_id, members.first_name, COUNT(*) AS borrows_count
FROM checkouts
JOIN members
ON checkouts.member_id = members.member_id
GROUP BY checkouts.member_id
ORDER BY COUNT(*) DESC
LIMIT 10
'''
print(pd.read_sql_query(query, conn)) #Who are the most active readers?

query = '''
SELECT checkouts.checkout_id, members.first_name, checkouts.checkout_date, members.neighborhood
FROM checkouts
JOIN members
ON checkouts.member_id = members.member_id
WHERE members.neighborhood = 'Maadi'
ORDER BY checkouts.checkout_date DESC
LIMIT -1 --no limit
OFFSET 10
'''
print(pd.read_sql_query(query, conn)) #What does a neighborhood's (Maadi) activity look like further back in time?



df_members = pd.read_sql_query('SELECT * FROM members', conn)
df_books = pd.read_sql_query('SELECT * FROM books', conn)
df_checkouts = pd.read_sql_query('SELECT * FROM checkouts', conn)

stage_1 = pd.merge(df_checkouts, df_members, on='member_id', how='left')

borrows_count = stage_1['member_id'].value_counts() #The combined result lets you see for each member how many books they've borrowed in total.
print(borrows_count)

print()

df_json = pd.read_json(r'c:\Users\HP\OneDrive\Desktop\download (1)')

combined_books = pd.merge(df_books, df_json, on='book_id', how='left') #combining all books information (from json and books sql table)
stage_2 = pd.merge(stage_1, combined_books, on='book_id', how='left')

tables = pd.read_html(r'c:\Users\HP\OneDrive\Desktop\download (2)')
df_html = tables[0]

print("Columns' names:", list(df_html.columns)) #the columns' names are written in a different format from the other dataframe
df_html.columns = ['member_id', 'book_id', 'checkout_date'] #fixing the columns' names

print()

stage_3 = pd.merge(df_html, df_members, on='member_id', how='left') #merging the html with members details to complete its data
stage_3 = pd.merge(stage_3, combined_books, on='book_id', how='left') #merging the html with books details to complete its data
df = pd.concat([stage_2, stage_3], ignore_index=True)

print(df)
df.to_csv('task1_combined_data.csv', index=False)
# level3_final_project_library.db
# level3_final_project_book_catalog.json
# level3_final_project_event_signup.html

    member_id first_name last_name  grade neighborhood membership_status  \
0        1001      Salma   Ibrahim    8.0        Maadi            Active   
1        1002      Fares     Saleh    9.0        Maadi            Active   
2        1003     Bassel    Hegazy    6.0        Maadi            Active   
3        1004      Fares     Wahba    7.0        Maadi          inactive   
4        1005    Youssef     Halim    9.0        Maadi            Active   
..        ...        ...       ...    ...          ...               ...   
75       1076       Dina     Wahba    7.0       Shubra            Active   
76       1077       Lina    Rashad    6.0       Shubra            Active   
77       1078     Habiba     Osman    7.0       Shubra          INACTIVE   
78       1079       Rana     Osman    8.0       Shubra            Active   
79       1080     Bassel     Wahba    9.0       Shubra            Active   

     join_date  
0   2023-04-05  
1         None  
2   2025-04-23  
3   2024-10-09  
4 

# Task 2

In [141]:
import pandas as pd

df = pd.read_csv('task1_combined_data.csv')
print(df.head())

print('==========')

# print('Missing values to number of rows ratio in percentage per column:')
# print(df.isna().sum() / len(df) * 100) #checking the missing values to total number of rows ratio in percentage per column

print('==========')

print(f"Duplicate rows (checkout_id): {df.duplicated(subset=['checkout_id']).sum()}") #There are a lot of nans in the checkout_id column which are considered duplicates, so we will check for duplicates in all columns.
print(f"Duplicate rows (all columns): {df.duplicated().sum()}") 
print(f"Missing values in checkout_id): {df['checkout_id'].isna().sum()}") #checking the missing values in the checkout_id column to make sure all the extra duplicates are due to the nans in this column

df = df.drop_duplicates() #dropping the duplicates

print('==========')

string_columns = ['first_name', 'last_name', 'neighborhood', 'membership_status', 'title', 'author', 'genre', 'publisher']
for column in string_columns:
    df[column] = df[column].str.strip().str.title()
    print(f"- Unique values in {column}: {df[column].unique()}") #No more consistency issues in the string columns

print(df.dtypes) #checking the datatypes of the columns, dates aren't datetime, and membership_status is object instead of category

df['membership_status'] = df['membership_status'].astype('category')

dates = ['checkout_date', 'return_date', 'join_date']
for column in dates:
    df[column] = pd.to_datetime(df[column], format='mixed') #converting the date columns to datetime format, errors='coerce' will convert any invalid date to NaT

print(df.describe()) #checking for problems in the numerical columns, No problems found

print('==========')

unregistered_members = df[~df['member_id'].isin(df_members['member_id'])]
print(unregistered_members)

df = df[df['member_id'].isin(df_members['member_id'])] #removing the invalid members from the dataframe


df.to_csv('task2_cleaned_data.csv', index=False)

   checkout_id  member_id  book_id checkout_date return_date first_name  \
0       9263.0       1047      517    2024-10-21  2024-11-07       Sara   
1       9340.0       1072      513    2025-08-24  2025-09-01       Seif   
2       9231.0       1053      523    2024-02-04  2024-02-16       Adam   
3       9129.0       1032      513    2025-06-21  2025-06-29       Nada   
4       9370.0       1079      511    2025-11-11  2025-12-03       Rana   

  last_name  grade neighborhood membership_status   join_date  \
0    Rashad    NaN   Heliopolis          Inactive  2024-06-25   
1      Zaki    9.0      Zamalek            Active  2025-10-21   
2    Shafik    9.0   Heliopolis            Active  2024-01-03   
3      Zaki    7.0    Nasr City            Active  2025-10-19   
4     Osman    8.0       Shubra            Active  2024-10-27   

                     title        author       genre  pages  publication_year  \
0  Shadows on the Corniche   Hani Nagati     Mystery    338            2015.0